First version. Only valid ISBNs obtained by removing invalid characters (but result not checked for actual existence). Only non-zero ratings. Only users and books that have at least 3 ratings. 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix

from load_data import load_data
from valid_test_select import valid_test_select, valid_test_select_per_user
from initialize_model import initialize_mu_b_c
from helpers import get_rmse, get_ndcg
from fit_model import (
    fit_model_no_UV,
    fit_model_full,
    fit_model_full_beta,
    fit_model_UV_only,
)

In [3]:
np.set_printoptions(linewidth=150, precision=6)  # 75, 8

# Loading Data

In [4]:
df = load_data()

In [5]:
df.agg(["min", "max", "nunique", "dtype", "count", "size"])

,userid,isbn,rating
min,2,0000001481,0
max,278854,9998150752,10
nunique,103371,328465,11
dtype,int32,object,int8
count,1135338,1135338,1135338
size,1135338,1135338,1135338


In [6]:
df = df[df.rating > 0]

In [7]:
df.agg(["min", "max", "nunique", "size"])

,userid,isbn,rating
min,8,000003827X,1
max,278854,9997555635,10
nunique,76331,179924,10
size,427261,427261,427261


# Creating Matrix Y

In [8]:
# Encoding userids and isbns as categories (integers 0 to N-1)

user_cats = df.userid.astype("category")
book_cats = df.isbn.astype("category")

In [9]:
# Creating sparse matrix and converting it to pd.DataFrame

Y_sparse = coo_matrix((df.rating.values, (user_cats.cat.codes, book_cats.cat.codes)))
Y = pd.DataFrame(Y_sparse.toarray())

In [10]:
# print("Y shape:", Y.shape)  # (76331, 179924)
# print("total entries:", (Y > 0).sum().sum())  # 427261
# print("avg number of ratings per user:", round((Y > 0).sum(axis=1).mean(), 2))  # 5.6
# print("avg number of ratings per book:", round((Y > 0).sum(axis=0).mean(), 2))  # 2.37

In [11]:
# Filtering out improbably active readers

# np.sort(np.partition((Y > 0).sum(axis=1), -20)[-20:])
Y = Y.loc[(Y > 0).sum(axis=1) < 2000, :]

In [12]:
# Filtering users and books with enough observations

min_rats = 5
old_rows, old_cols = Y.shape
while True:
    Y_pos = Y > 0  # type: ignore
    user_rats = Y_pos.sum(axis=1)  # type: ignore
    book_rats = Y_pos.sum(axis=0)  # type: ignore
    new_rows = (user_rats >= min_rats).sum()
    new_cols = (book_rats >= min_rats).sum()
    # print(new_rows, new_cols)
    Y = Y.loc[user_rats >= min_rats, book_rats >= min_rats]  # type: ignore
    if (old_rows == new_rows) and (old_cols == new_cols):
        break
    old_rows, old_cols = new_rows, new_cols

del Y_sparse, Y_pos, user_rats, book_rats, new_rows, new_cols, old_rows, old_cols
gc.collect()

40

In [13]:
print("Y shape:", Y.shape)  # (6916, 8903) |3: (14903, 22661)
print("total entries:", (Y > 0).sum().sum())  # 113019 |3: 182612
print("avg # ratings per user:", f"{(Y > 0).sum(axis=1).mean():.2f}")  # 16.34 |3: 12.25
print("median # ratings per user:", f"{(Y > 0).sum(axis=1).median():.2f}")  # 9.00
print("avg # ratings per book:", f"{(Y > 0).sum(axis=0).mean():.2f}")  # 12.69 |3: 8.06
print("median # ratings per book:", f"{(Y > 0).sum(axis=0).median():.2f}")  #

Y shape: (6916, 8903)
total entries: 113019
avg # ratings per user: 16.34
median # ratings per user: 9.00
avg # ratings per book: 12.69
median # ratings per book: 8.00


In [14]:
# Converting Y to numpy array

Y_columns = Y.columns.copy()
Y = Y.to_numpy()

# Selecting validation and test sets

In [15]:
# Y_train, valid_data, test_data = valid_test_select(Y, 20_000, 20_000, seed=100)
Y_train, valid_data, test_data = valid_test_select_per_user(Y, seed=100)

In [16]:
print("Y_train shape:", Y_train.shape)  # (14903, 22661)
print("total train entries:", (Y_train > 0).sum().sum())  # 142612
print("avg. # train entries/user:", f"{(Y_train > 0).sum(axis=1).mean():.2f}")  # 9.57
print("avg. # train entries/book:", f"{(Y_train > 0).sum(axis=0).mean():.2f}")  # 6.29
print("number of validation samples:", valid_data.shape[0])
print("number of test samples:", test_data.shape[0])  # type: ignore

Y_train shape: (6916, 8903)
total train entries: 83288
avg. # train entries/user: 12.04
avg. # train entries/book: 9.36
number of validation samples: 20407
number of test samples: 9324


# ALS

In [17]:
Y_csr = csr_matrix(Y_train)
Y_csc = Y_csr.tocsc()

## Model with just a training global mean

In [18]:
mu, _, _ = initialize_mu_b_c(Y_train)
print("test RMSE of model=mu:", get_rmse(mu, test_data))
preds_ndcg = np.full(Y_train.shape, mu)
print("test NDCG of model=mu:", get_ndcg(preds_ndcg, Y_train, test_data, 5))
# min_rats=3: test rmse of model=mu: 1.8087624016357353

test RMSE of model=mu: 1.7874936120810423
test NDCG of model=mu: 8.395544798110088e-05


## Model with only biases

In [19]:
mu, b, c = initialize_mu_b_c(Y_train)
mu_bias, b_bias, c_bias = fit_model_no_UV(
    Y_train, Y_csr, Y_csc, mu, b, c, 3.8, valid_data, 1e-5, info=1
)

results: counter =  20, max_norm_diff =    0.05, valid_rmse = 1.600946, valid_ndcg = 0.000612, valid_rmse_diff = 9.6e-06, valid_ndcg_diff = 0      


In [20]:
test_preds_rmse = np.clip(
    mu_bias + b_bias[test_data.rows] + c_bias[test_data.cols], 1, 10  # type: ignore
)
test_preds_ndcg = mu_bias + b_bias[:, None] + c_bias[None, :]
test_rmse_biases = get_rmse(test_preds_rmse, test_data)
test_ndcg_biases = get_ndcg(test_preds_ndcg, Y_train, test_data, 5)
print("test RMSE of a tuned bias-only model:", test_rmse_biases)
print("test NDCG of a tuned bias-only model:", test_ndcg_biases)
# min_rats = 3: rmse = 1.5616776141641613

test RMSE of a tuned bias-only model: 1.5845852180517217
test NDCG of a tuned bias-only model: 0.0005618910102909354


## Full model - tuning

In [38]:
# k, l_bias, l_fact
to_try = [
    [384, 4, 9.5],
    [384, 3.9, 9.5],
    [384, 4.1, 9.5],
    [384, 4, 9],
    [384, 4, 10],
]

In [39]:
for k, l_b, l_f in to_try:
    gc.collect()
    print(f"{k = :>3}, {l_b = :>5.2f}, {l_f = :>5.2f}", end=", ")
    mu, b, c = initialize_mu_b_c(Y_train)
    rng = np.random.default_rng(seed=123)
    U = rng.normal(0, 0.5, size=(Y.shape[0], k))
    V = rng.normal(0, 0.5, size=(Y.shape[1], k))
    mu, b, c, U, V = fit_model_full(
        Y_train, Y_csr, Y_csc, mu, b, c, U, V, l_b, l_f, valid_data, 1e-5, info=1
    )

k = 384, l_b =  4.00, l_f =  9.50, results: counter =  13, max_norm_diff =    1.46, valid_rmse = 1.599096, valid_ndcg = 0.001391
k = 384, l_b =  3.90, l_f =  9.50, results: counter =  13, max_norm_diff =    1.45, valid_rmse = 1.598911, valid_ndcg = 0.001378
k = 384, l_b =  4.10, l_f =  9.50, results: counter =  12, max_norm_diff =    1.64, valid_rmse = 1.599312, valid_ndcg = 0.001380
k = 384, l_b =  4.00, l_f =  9.00, results: counter =  11, max_norm_diff =    1.98, valid_rmse = 1.599198, valid_ndcg = 0.001494
k = 384, l_b =  4.00, l_f = 10.00, results: counter =  14, max_norm_diff =    1.22, valid_rmse = 1.599047, valid_ndcg = 0.001312


## Full model - testing

In [ ]:
# k, l_bias, l_fact
to_try = [
    [384, 4, 9.5],
]

# k = 384, l_b =  4.00, l_f =  9.50, results: counter =  13, max_norm_diff =    1.46, valid_rmse = 1.599096, valid_ndcg = 0.001391
#    test rmse of a full model with k = 384: 1.5839539674384298
#    test rmse of a full model with k = 384: 0.0011819987943781027

In [ ]:
print("test rmse of a tuned model with only biases:", test_rmse_biases)

for k, l_b, l_f in to_try:
    print(f"{k = :>3}, {l_b = :>4.2f}, {l_f = :>4.2f}", end=", ")
    mu, b, c = initialize_mu_b_c(Y_train)
    rng = np.random.default_rng(seed=123)
    U = rng.normal(0, 0.5, size=(Y.shape[0], k))
    V = rng.normal(0, 0.5, size=(Y.shape[1], k))
    mu, b, c, U, V = fit_model_full(
        Y_train, Y_csr, Y_csc, mu, b, c, U, V, l_b, l_f, valid_data, 1e-5, info=1
    )
    biases = mu + b[test_data.rows] + c[test_data.cols]  # type: ignore
    preds_rmse = np.clip(
        biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10  # type: ignore
    )
    print(
        f"   test rmse of a full model with {k = :>3}:", get_rmse(preds_rmse, test_data)
    )
    preds_ndcg = mu + b[:, None] + c[None, :] + U @ V.T
    print(
        f"   test rmse of a full model with {k = :>3}:",
        get_ndcg(preds_ndcg, Y_train, test_data, 5),
    )

test rmse of a tuned model with only biases: 1.5845852180517217
k = 384, l_b = 4.00, l_f = 9.50, results: counter =  13, max_norm_diff =    1.46, valid_rmse = 1.599096, valid_ndcg = 0.001391
   test rmse of a full model with k = 384: 1.5839539674384298
     test rmse of a full model with k = 384: 0.0011819987943781027


## Full model with beta - tuning

In [ ]:
# k, l_bias, l_fact, beta
to_try = []

# k = 384, l_b = 10000, l_f = 11.00, beta = 0.1,   results: counter =   2, max_norm_diff =   30.94, valid_rmse = 1.786641, valid_ndcg = 0.019150
#  test rmse of a full model with k = 384: 1.767088304942898
#  test rmse of a full model with k = 384: 0.012715877535559047

In [ ]:
for k, l_b, l_f, beta in to_try:
    gc.collect()
    gc.collect()
    print(f"{k = :>3}, {l_b = :>5.2f}, {l_f = :>5.2f}, {beta = }", end=", ")
    mu, b, c = initialize_mu_b_c(Y_train)
    rng = np.random.default_rng(seed=123)
    U = rng.normal(0, 0.5, size=(Y.shape[0], k))
    V = rng.normal(0, 0.5, size=(Y.shape[1], k))
    mu, b, c, U, V = fit_model_full_beta(
        Y_train, Y_csr, Y_csc, mu, b, c, U, V, l_b, l_f, beta, valid_data, 1e-5, info=1
    )

## Full model with beta - testing

In [36]:
# k, l_bias, l_fact
to_try = [
    [384, 10000, 11, 0.1],
]

In [37]:
print("test RMSE of a tuned model with only biases:", test_rmse_biases)
print("test NDCG of a tuned model with only biases:", test_ndcg_biases)

for k, l_b, l_f, beta in to_try:
    print(f"{k = :>3}, {l_b = :>5.2f}, {l_f = :>5.2f}, {beta = }", end=", ")
    mu, b, c = initialize_mu_b_c(Y_train)
    rng = np.random.default_rng(seed=123)
    U = rng.normal(0, 0.5, size=(Y.shape[0], k))
    V = rng.normal(0, 0.5, size=(Y.shape[1], k))
    mu, b, c, U, V = fit_model_full_beta(
        Y_train, Y_csr, Y_csc, mu, b, c, U, V, l_b, l_f, beta, valid_data, 1e-5, info=1
    )
    biases = mu + b[test_data.rows] + c[test_data.cols]  # type: ignore
    preds_rmse = np.clip(
        biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10  # type: ignore
    )
    print(
        f"     test rmse of a full model with {k = :>3}:",
        get_rmse(preds_rmse, test_data),
    )
    preds_ndcg = mu + b[:, None] + beta * c[None, :] + U @ V.T
    print(
        f"     test rmse of a full model with {k = :>3}:",
        get_ndcg(preds_ndcg, Y_train, test_data, 5),
    )

test RMSE of a tuned model with only biases: 1.5845852180517217
test NDCG of a tuned model with only biases: 0.0005618910102909354
k = 384, l_b = 10000.00, l_f = 11.00, beta = 0.1, results: counter =   2, max_norm_diff =   30.94, valid_rmse = 1.786641, valid_ndcg = 0.019150
     test rmse of a full model with k = 384: 1.767088304942898
     test rmse of a full model with k = 384: 0.012715877535559047


## UV-only model - tuning

In [ ]:
# k, l_fact
to_try = []

# k = 512, l_f = 17  , results: counter =   2, max_norm_diff =  123.15, valid_rmse = 5.986533, valid_ndcg = 0.033795
# k = 512, l_f = 18  , results: counter =   2, max_norm_diff =  121.85, valid_rmse = 5.990860, valid_ndcg = 0.033963
# k = 512, l_f = 19  , results: counter =   2, max_norm_diff =  120.62, valid_rmse = 5.997183, valid_ndcg = 0.033952
#  test rmse of a full model with k = 512: 6.0071888058443434
#  test rmse of a full model with k = 512: 0.026537165204014217

# k = 384, l_f = 20,   results: counter =   2, max_norm_diff =  119.49, valid_rmse = 6.020871, valid_ndcg = 0.032450
# k = 384, l_f = 21,   results: counter =   2, max_norm_diff =  118.61, valid_rmse = 6.027936, valid_ndcg = 0.032480
# k = 384, l_f = 22,   results: counter =   2, max_norm_diff =  117.78, valid_rmse = 6.036463, valid_ndcg = 0.032133
#  test rmse of a full model with k = 384: 6.038851200960989
#  test rmse of a full model with k = 384: 0.025499948861290044

# k = 128, l_f =   32, results: counter =   2, max_norm_diff =  117.20, valid_rmse = 6.215150, valid_ndcg = 0.026227
# k = 128, l_f =   33, results: counter =   2, max_norm_diff =  116.35, valid_rmse = 6.229260, valid_ndcg = 0.026478
# k = 128, l_f =   34, results: counter =   2, max_norm_diff =  115.52, valid_rmse = 6.243651, valid_ndcg = 0.026264
#  test rmse of a full model with k = 128: 6.2464139948513235
#  test rmse of a full model with k = 128: 0.020542998853865815

# k =   4, l_f = 124, results: counter =   3, max_norm_diff =   32.44, valid_rmse = 6.890795, valid_ndcg = 0.012209
# k =   4, l_f = 125, results: counter =   3, max_norm_diff =   32.09, valid_rmse = 6.895400, valid_ndcg = 0.012216
# k =   4, l_f = 126, results: counter =   3, max_norm_diff =   31.73, valid_rmse = 6.899793, valid_ndcg = 0.012031
#  test rmse of a full model with k =   4: 6.934749631661739
#  test rmse of a full model with k =   4: 0.0093796218698342

In [ ]:
for k, l_f in to_try:
    gc.collect()
    print(f"{k = :<3}, {l_f = :<4}", end=", ")
    rng = np.random.default_rng(seed=123)
    U = rng.normal(0, 0.5, size=(Y.shape[0], k))
    V = rng.normal(0, 0.5, size=(Y.shape[1], k))
    U, V = fit_model_UV_only(Y_train, Y_csr, Y_csc, U, V, l_f, valid_data, 1e-5, info=2)

k = 1024, l_f = 15  , original:                  valid rmse = 5.769863, valid ndcg = 0.000420
         counter =   1, max_norm_diff = 1412.41, valid_rmse = 6.915773, valid_ndcg = 0.029598
         counter =   2, max_norm_diff =  133.84, valid_rmse = 5.785185, valid_ndcg = 0.030002
         counter =   3, max_norm_diff =   92.28, valid_rmse = 4.489085, valid_ndcg = 0.017043
                        valid_ndcg_diff is under thresh, patience=1
         counter =   4, max_norm_diff =   60.72, valid_rmse = 3.870329, valid_ndcg = 0.014394
                        valid_ndcg_diff is under thresh, patience=2
         counter =   5, max_norm_diff =   36.75, valid_rmse = 3.631150, valid_ndcg = 0.013038
                        valid_ndcg_diff is under thresh, patience=3
results: counter =   2, max_norm_diff =  133.84, valid_rmse = 5.785185, valid_ndcg = 0.030002


## UV-only model - testing

In [ ]:
to_try = [[512, 18]]

In [ ]:
print("test RMSE of a tuned model with only biases:", test_rmse_biases)
print("test NDCG of a tuned model with only biases:", test_ndcg_biases)

for k, l_f in to_try:
    print(f"{k = :>3}, {l_f = :>4}", end=", ")
    rng = np.random.default_rng(seed=123)
    U = rng.normal(0, 0.5, size=(Y.shape[0], k))
    V = rng.normal(0, 0.5, size=(Y.shape[1], k))
    U, V = fit_model_UV_only(Y_train, Y_csr, Y_csc, U, V, l_f, valid_data, 1e-5, info=1)
    preds_rmse = np.clip(np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)  # type: ignore
    print(
        f"     test rmse of a full model with {k = :>3}:",
        get_rmse(preds_rmse, test_data),
    )
    preds_ndcg = U @ V.T
    print(
        f"     test rmse of a full model with {k = :>3}:",
        get_ndcg(preds_ndcg, Y_train, test_data, 5),
    )

test RMSE of a tuned model with only biases: 1.5845852180517217
test NDCG of a tuned model with only biases: 0.0005618910102909354
k = 512, l_f =   18, results: counter =   2, max_norm_diff =  121.85, valid_rmse = 5.990860, valid_ndcg = 0.033963
     test rmse of a full model with k = 512: 6.0071888058443434
     test rmse of a full model with k = 512: 0.026537165204014217
